# CONCLAVE — Phase 1 Reference Notebook

This is a **reference / options catalog**, not a run-and-forget notebook: every normalization
method, sampling mode, and dimensionality-reduction method is shown separately with its
hyperparameters explained, so you can see what each one does on *your* data before picking a
combination for a real run.

All examples below were verified against a real slice of `Melanoma.csv` (not synthetic data) —
every method shown here is confirmed working. Two known gaps as of this notebook:

- **FlowSOM / DepecheR** clustering need an external R script (`flowsom_rscript=...` /
  `depeche_rscript=...`) that isn't bundled with CONCLAVE yet — not covered here.
- **PaCMAP** is an *optional* dependency (`pip install conclave[dr]` or `pip install pacmap`
  directly) — everything else is a core dependency.

For an actual end-to-end run, use the companion `CONCLAVE_Phase1.ipynb` notebook instead — this
one is for exploring what each option does.

## Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from conclave.phase1.normalization import normalize_markers
from conclave.phase1.sampling import sample_umap_tiles
from conclave.phase1.visualization import run_dr

pd.set_option("display.max_columns", 50)

In [ ]:
CSV_PATH = "Melanoma.csv"
CSV_NROWS = 5000  # keep small here since this notebook runs *every* method for comparison

df = pd.read_csv(CSV_PATH, nrows=CSV_NROWS)

MARKERS = [
    'CD34', 'CD31', 'CD141', 'PNAd', 'CD25', 'CD14', 'CD1c', 'CK', 'CD21',
    'FoxP3', 'CD23', 'GRB7', 'CD1A', 'Podoplanin', 'CD138', 'CD248', 'CD64', 'CD163',
    'Pax5', 'IRF8', 'CD20', 'CD8', 'CD303', 'LYZ', 'CD16', 'CD2', 'HLADR', 'IRF4', 'CD5',
    'CD79a', 'CD68', 'CD3', 'CD4', 'CD27', 'PRDM1', 'MELANA', 'S100B',
]
SAMPLE_COLS = ["ID"]

print(f"{df.shape[0]:,} cells x {len(MARKERS)} markers, {df[SAMPLE_COLS[0]].nunique()} sample(s)")

---
## 1. Normalization — `normalize_markers()`

Signature:

```python
normalize_markers(df, markers, method="z-score", sample_cols=None,
                   clip=5.0, q=0.99, scale=1e4, logger=None)
```

| `method` | What it does | Key hyperparameters |
|---|---|---|
| `None` | No transform, just coerces to numeric/fills NaN with 0 | — |
| `"z-score"` | `(x - mean) / std`, then clipped to `[-clip, +clip]` | `clip` (default 5.0; set `None` to disable clipping) |
| `"lognorm"` | Per-cell total-count normalize to `scale`, then `log1p` (CyCIF/CITE-seq-style) | `scale` (default 1e4) |
| `"minmax"` | Winsorize to the `[1-q, q]` quantiles, then rescale to `[0, 1]` | `q` (default 0.99) |
| `"iqr-zscore"` | Winsorize to Tukey fences `[Q1-1.5*IQR, Q3+1.5*IQR]`, then standardize (no further clipping) | — |
| `"iqr-minmax"` | Same Tukey-fence Winsorization, then rescale to `[0, 1]` using the winsorized range | — |

The two IQR-based methods use each marker's own interquartile range to decide what counts as an
outlier, rather than a fixed quantile (`minmax`) or a fixed z-score cutoff (`z-score`) -- often a
more robust default when different markers have very different outlier behavior.

**`sample_cols`**: if set (e.g. `["ID"]`), normalization statistics (mean/std, quantiles, etc.)
are computed *separately within each sample/slide* rather than pooled — recommended when you
have batch effects across slides. Set to `None` to pool everything as one group.

Below: all 6 methods run on the same data, both pooled and per-sample, so you can see the effect
directly.

In [ ]:
results = {}

for method in [None, "z-score", "lognorm", "minmax", "iqr-zscore", "iqr-minmax"]:
    for sample_cols, label in [(None, "pooled"), (SAMPLE_COLS, "per-sample")]:
        out, report = normalize_markers(df, MARKERS, method=method, sample_cols=sample_cols)
        key = f"{method}_{label}"
        results[key] = out
        vals = out[MARKERS].values
        print(f"{str(method):10s} | {label:10s} | mean={vals.mean():+7.3f}  std={vals.std():6.3f}  "
              f"min={vals.min():+7.3f}  max={vals.max():7.3f}")

In [ ]:
# Visual comparison: distribution of a single marker under each method
fig, axes = plt.subplots(1, 6, figsize=(22, 3), sharey=False)
marker_to_plot = MARKERS[0]

for ax, method in zip(axes, [None, "z-score", "lognorm", "minmax", "iqr-zscore", "iqr-minmax"]):
    out = results[f"{method}_pooled"]
    ax.hist(out[marker_to_plot], bins=50)
    ax.set_title(f"{method}")
    ax.set_xlabel(marker_to_plot)

plt.tight_layout()
plt.show()

**Z-score clipping demo** — `clip` controls how aggressively outliers get pulled in.
Set `clip=None` to disable clipping entirely (raw z-scores, unbounded).

In [ ]:
for clip_val in [None, 1.0, 3.0, 5.0]:
    out, _ = normalize_markers(df, MARKERS, method="z-score", clip=clip_val)
    vals = out[MARKERS].values
    print(f"clip={str(clip_val):5s} -> min={vals.min():+7.3f}  max={vals.max():7.3f}")

---
## 2. Sampling — `sample_umap_tiles()`

Signature:

```python
sample_umap_tiles(df, markers, sample_size=None, mode="stratified-notproportional",
                   n_tiles_per_axis=4, random_state=42, umap_params=None,
                   use_gpu=False, gpu_pca_components=50, logger=None)
```

Phase 1 clusters on a subsample rather than the full dataset (for speed, and so rare cell
populations aren't drowned out). This subsample is chosen via a quick 2D UMAP embedding, tiled
into an `n_tiles_per_axis x n_tiles_per_axis` grid.

| `mode` | What it does |
|---|---|
| `"none"` / `sample_size=None` | No subsampling — returns the full dataset |
| `"random"` | Plain uniform random sample of `sample_size` cells |
| `"stratified-proportional"` | Samples from each UMAP tile in proportion to that tile's size (denser regions contribute more cells, same relative balance as the full data) |
| `"stratified-notproportional"` | Samples roughly equally from *every* tile regardless of tile size (boosts rare populations relative to their true frequency — the manuscript's default) |

**Key hyperparameters:**
- `sample_size`: target subsample size
- `n_tiles_per_axis`: grid resolution for stratification (4 → 16 tiles); ignored for `"random"`/`"none"`
- `umap_params`: dict, e.g. `{"n_neighbors": 15, "min_dist": 0.1}` — controls the UMAP embedding used only for tiling, not your final clustering embedding
- `use_gpu`: tries cuML, falls back to PyTorch-GPU-PCA+CPU-UMAP, falls back to plain CPU — safe to leave `True` even without a GPU, it just falls back

In [ ]:
TARGET_SAMPLE_SIZE = 800

for mode in ["none", "random", "stratified-proportional", "stratified-notproportional"]:
    t0 = time.time()
    out = sample_umap_tiles(df, MARKERS, sample_size=TARGET_SAMPLE_SIZE, mode=mode,
                             n_tiles_per_axis=4, random_state=42)
    dt = time.time() - t0
    print(f"{mode:28s} -> {len(out):5,} cells  ({dt:5.1f}s)")

**Proportional vs. non-proportional, visualized** — non-proportional should look more
"evenly spread" across UMAP space; proportional should mirror the original density.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, mode in zip(axes, ["stratified-proportional", "stratified-notproportional"]):
    out = sample_umap_tiles(df, MARKERS, sample_size=TARGET_SAMPLE_SIZE, mode=mode,
                             n_tiles_per_axis=4, random_state=42)
    ax.scatter(out["umap_x"], out["umap_y"], s=4, alpha=0.6)
    ax.set_title(f"{mode}\n({len(out)} cells)")

plt.tight_layout()
plt.show()

**`n_tiles_per_axis` effect** — more tiles = finer stratification = better coverage of small
populations, but more overhead.

In [ ]:
for n_tiles in [2, 4, 6]:
    t0 = time.time()
    out = sample_umap_tiles(df, MARKERS, sample_size=TARGET_SAMPLE_SIZE,
                             mode="stratified-notproportional",
                             n_tiles_per_axis=n_tiles, random_state=42)
    print(f"n_tiles_per_axis={n_tiles} -> {n_tiles*n_tiles:3d} tiles, {len(out)} cells sampled ({time.time()-t0:.1f}s)")

---
## 3. Dimensionality Reduction — `run_dr()`

Signature:

```python
run_dr(df, markers, method=None, n_components=15, random_state=42,
       umap_params=None, pacmap_params=None, tsne_params=None, logger=None)
```

Clustering can run directly on the (normalized) marker space (`method=None`, the manuscript's
default) or on a reduced embedding first.

| `method` | Hyperparameters | Notes |
|---|---|---|
| `None` | — | Uses marker space directly, `n_components` is ignored |
| `"pca"` | `n_components` | Fast, linear, deterministic |
| `"umap"` | `n_components`, `umap_params={"n_neighbors", "min_dist", "metric"}` | Nonlinear, preserves both local and some global structure |
| `"pacmap"` | `n_components`, `pacmap_params` | Nonlinear, tends to preserve global structure better than UMAP; **optional dependency** |
| `"tsne"` | `n_components` (**capped at 3** — sklearn's fast solver doesn't go higher, and the manuscript restricts t-SNE to 3D), `tsne_params={"perplexity", "learning_rate", "init"}` | Nonlinear, local-structure-focused; **slowest option by far** — took ~165s for 5,000 cells in testing vs. ~6s for UMAP |

Requesting `n_components` > 3 for t-SNE doesn't error — it's silently clipped to 3, with a
warning logged, and the DR result's metadata records both the requested and actual value.

In [ ]:
df_small = df.sample(n=min(1500, len(df)), random_state=42)  # keep tsne fast for this demo

dr_results = {}
for method, ncomp in [(None, 15), ("pca", 10), ("umap", 3), ("pacmap", 3), ("tsne", 3)]:
    t0 = time.time()
    Xr, info = run_dr(df_small, MARKERS, method=method, n_components=ncomp, random_state=42)
    dt = time.time() - t0
    dr_results[str(method)] = Xr
    print(f"{str(method):8s} -> shape={Xr.shape}  ({dt:5.1f}s)  info={ {k:v for k,v in info.items() if k!='explained_variance_ratio'} }")

**t-SNE's 3-component cap, explicitly demonstrated:**

In [ ]:
Xr, info = run_dr(df_small, MARKERS, method="tsne", n_components=15, random_state=42)
print(f"Requested n_components=15 -> got shape {Xr.shape}")
print(f"info: {info}")

**2D visual comparison** (using the first 2 dims of each method's embedding, PCA'd down
further where needed, just for a quick visual sanity check — not meant to substitute for proper
downstream analysis):

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, Xr) in zip(axes, dr_results.items()):
    X2 = Xr[:, :2] if Xr.shape[1] >= 2 else PCA(n_components=2).fit_transform(Xr)
    ax.scatter(X2[:, 0], X2[:, 1], s=3, alpha=0.5)
    ax.set_title(name)
plt.tight_layout()
plt.show()

**UMAP hyperparameters** (`n_neighbors`, `min_dist`) — same idea applies to the UMAP used
inside sampling (`umap_params` there) and PaCMAP's equivalent knobs:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, n_neighbors in zip(axes, [5, 15, 50]):
    Xr, _ = run_dr(df_small, MARKERS, method="umap", n_components=2, random_state=42,
                   umap_params={"n_neighbors": n_neighbors, "min_dist": 0.1})
    ax.scatter(Xr[:, 0], Xr[:, 1], s=3, alpha=0.5)
    ax.set_title(f"n_neighbors={n_neighbors}")
plt.tight_layout()
plt.show()

---
## Summary: what to actually pick

There's no universally-correct combination — but as a starting point matching the CONCLAVE
manuscript's methodology:

```python
normalization = "z-score"                          # clip=5.0
sampling = "stratified-notproportional"             # boosts rare populations
dr_method = None                                     # cluster directly on marker space
cluster_methods = ("phenograph", "flowsom", "kmeans")  # phenograph_k=25
```

See `CONCLAVE_Phase1.ipynb` for a runnable version of this.